# NYC Taxi Demand Training

이 노트북은 A-Eye의 첫 학습 흐름을 보여준다. NYC TLC 5분 단위 실측 택시 픽업 라벨로 `t -> t+5min` 수요 예측 모델을 학습하고, 강남 데이터에는 정확한 택시 호출 라벨이 없으므로 상대 수요 heatmap prior로만 적용한다.

## 0. Locate Repo

VS Code Jupyter는 현재 작업 디렉토리를 `A-Eye/`로 잡을 때도 있고 `A-Eye/notebooks/`로 잡을 때도 있다. 아래 셀은 어느 쪽이어도 repo root로 이동한다.

In [ ]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = next(
    path for path in candidates
    if (path / 'scripts' / 'train_nyc_baseline_model.py').exists()
)
os.chdir(repo_root)
print('repo root:', Path.cwd())

## 1. Train Baseline

아래 셀은 같은 repo의 재현 가능한 학습 스크립트를 실행한다. 이 baseline은 NYC 15개월 데이터에서 150만 row를 샘플링해 학습하고, 2026년 3월 전체를 테스트셋으로 평가한다.

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python3 scripts/train_nyc_baseline_model.py

## 2. Load Metrics

In [ ]:
import json
from pathlib import Path

metrics_path = Path('data/processed/model/nyc_baseline_metrics.json')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
metrics['metrics'], metrics['top5_overlap']

## 3. Build Context Features

NYC taxi 5분 라벨에 날씨, 미국 공휴일, Manhattan 지하철 시간대 ridership을 붙인다.

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python3 scripts/build_nyc_context_features.py

## 4. Train Context-Aware Model

context feature를 포함한 Ridge 모델을 학습하고, 기존 baseline과 비교한다.

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python3 scripts/train_nyc_context_model.py

## 5. Load Context Metrics

In [ ]:
context_path = Path('data/processed/model/nyc_context_metrics.json')
context_metrics = json.loads(context_path.read_text(encoding='utf-8'))
context_metrics['metrics'], context_metrics['comparison']

## 6. Generate Figures

학습 결과를 발표/보고서에 넣을 수 있도록 SVG 그래프를 생성한다.

In [ ]:
!PYTHONDONTWRITEBYTECODE=1 python3 scripts/plot_nyc_baseline_results.py

## 7. View Figures

In [ ]:
from IPython.display import SVG, display

figure_paths = [
    'data/processed/model/figures/nyc_baseline_metrics.svg',
    'data/processed/model/figures/nyc_context_comparison.svg',
    'data/processed/model/figures/nyc_hourly_pattern.svg',
    'data/processed/model/figures/nyc_actual_vs_predicted.svg',
    'data/processed/model/figures/nyc_error_by_zone_type.svg',
]

for path in figure_paths:
    display(SVG(filename=path))

## 8. Interpretation

- NYC에서는 실제 5분 단위 택시 픽업 정답 라벨이 있으므로 모델 학습/평가가 가능하다.
- weather, holiday, subway ridership을 택시 수요 라벨에 시간 단위로 join했다.
- 5분 horizon에서는 최근 택시 수요 lag가 가장 강하고, 추가 context 중에서는 subway ridership이 가장 큰 설명 신호로 잡힌다.
- 강남 CSV는 공공 feature table이고 정확한 카카오택시 호출 수/좌표 라벨이 아니므로, exact count 예측이 아니라 상대 수요 heatmap으로 설명한다.
- 다음 단계는 이 baseline을 XGBoost/LSTM/ConvLSTM 등과 비교하는 것이다.